In [20]:
import torch
import numpy as np
import glob
from PIL import Image
from tqdm import tqdm

category2code = {"asphalt": 0, "ceramic": 1, "concrete": 2, "fabric": 3, "foliage": 4,
                            "food": 5, "glass": 6, "metal": 7, "paper": 8, "plaster": 9, "plastic": 10,
                            "rubber": 11, "soil": 12, "stone": 13, "water": 14, "wood": 15}

gts = glob.glob("matbase/masks_png/*.png")

In [21]:
materials = glob.glob('materials_numpy/*.npy')
materials = [torch.from_numpy(np.load(m)).float() for m in materials]
code2material = {v: k for k, v in zip(materials, category2code.values())}

def convert_target_to_material(target):
    """
    Converts the target image to a material image.
    Params:
        target -> Hyperspectral image (height, width, 1)
    Returns:
        material_image -> Material image (height, width, wavelength)
    """
    height, width, _ = target.shape
    target_flat = target.view(-1)
    material_flat = torch.stack([
        code2material[val.item()] if val.item() != 255 else torch.zeros(31)
        for val in target_flat
    ])
    material_image = material_flat.view(height, width, -1)
    return material_image

for gt in tqdm(gts):
    mask = np.array(Image.open(gt))
    mask = torch.from_numpy(mask).unsqueeze(-1).float()
    mask = convert_target_to_material(mask)
    torch.save(mask, "materials/" + gt.split("/")[-1].replace(".png", ".pt"))


100%|██████████| 5151/5151 [2:22:07<00:00,  1.66s/it]  


In [17]:
#load
torch.load("materials/COCO_train2014_000000203859.pt").shape

torch.Size([512, 482, 31])